# Assignment 1 — Build a Custom Missing-Value Imputer

**Course:** Feature Engineering & MLOps

**Topic:** Custom Imputer Class (Missing Value Handling)

**Dataset:** `student_performance_raw.csv` (PrepEdge coaching-institute dataset, Unit 1 — Session 4)

This notebook implements `CustomImputer`, a scikit-learn-compatible transformer that fills missing
values following strict train/test discipline, applies it to the dataset, verifies it, sanity-checks
it against `SimpleImputer`, and answers the reflection questions.

In [1]:
import numpy as np
import pandas as pd
import pandas.api.types as ptypes

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 20)
np.random.seed(42)

## 1. The `CustomImputer` Class

Design decisions, up front:

- **Column-type detection** uses `pandas.api.types.is_numeric_dtype`, so numeric dtypes (int/float) are
  imputed with a mean/median and everything else (strings, categoricals, and the free-text
  `feedback_text` column) is imputed with the mode (`most_frequent`). The assignment only asks the class
  to distinguish *numeric vs. categorical*, so free text is treated as a categorical column — filling it
  with its single most common phrase is a reasonable (if crude) default for a generic imputer.
- **No leakage by construction**: `fit()` is the *only* place any statistic is computed. `transform()`
  only ever looks up values already stored in `self.fill_values_`; it never recomputes anything from the
  data it is given, so it behaves identically whether it's transforming the training split, the test
  split, or brand-new data at inference time.
- **Missing indicators are computed from the training data's missingness pattern** (`self.indicator_cols_`,
  set in `fit`), but the indicator *values* in `transform()` reflect whichever split is actually passed in
  — e.g. a row in the test set is flagged `1` if *that row* was missing, not if the column was missing in
  training.

In [ ]:
class CustomImputer(BaseEstimator, TransformerMixin):
    """
    A custom missing-value imputer that follows scikit-learn's fit/transform
    estimator API (BaseEstimator + TransformerMixin).

    For every column seen during `fit`, a single fill value is learned
    (mean/median for numeric columns, the mode for categorical/text columns)
    and stored. `transform` only ever reuses those stored values -- it never
    recomputes statistics -- which is what guarantees no leakage between
    train and test splits.

    Parameters
    ----------
    numeric_strategy : {'mean', 'median'}, default='median'
        Statistic used to fill missing values in numeric columns, unless
        overridden for a specific column via `column_overrides`.
    categorical_strategy : {'most_frequent'}, default='most_frequent'
        Strategy used to fill missing values in categorical/text columns,
        unless overridden for a specific column via `column_overrides`.
    add_missing_indicator : bool, default=True
        If True, `transform` adds one binary column named
        '<column>_was_missing' for every column that had at least one
        missing value in the training data.
    column_overrides : dict or None, default=None
        Optional per-column strategy override, e.g.
        {'weekly_study_hours': 'mean', 'income_bracket': 'most_frequent'}.
        A key's value replaces `numeric_strategy` / `categorical_strategy`
        for that one column only. (Bonus feature -- Section 7 of Task 4.)

    Attributes
    ----------
    fill_values_ : dict
        column name -> fill value learned during `fit`.
    numeric_cols_ : list of str
        Columns auto-detected as numeric during `fit`.
    categorical_cols_ : list of str
        Columns auto-detected as categorical/text during `fit`.
    indicator_cols_ : list of str
        Columns that had at least one missing value in the training data
        (these are the columns that get a `_was_missing` flag).
    feature_names_in_ : list of str
        Column names seen during `fit`, in order.
    n_features_in_ : int
        Number of columns seen during `fit`.
    """

    def __init__(self, numeric_strategy='median', categorical_strategy='most_frequent',
                 add_missing_indicator=True, column_overrides=None):
        # scikit-learn convention: __init__ must store every constructor
        # argument unmodified as an attribute of the same name, so that
        # get_params()/set_params()/clone() work correctly.
        self.numeric_strategy = numeric_strategy
        self.categorical_strategy = categorical_strategy
        self.add_missing_indicator = add_missing_indicator
        self.column_overrides = column_overrides

    def _strategy_for(self, col, is_numeric):
        """Resolve which strategy string applies to `col`, respecting column_overrides."""
        if self.column_overrides and col in self.column_overrides:
            return self.column_overrides[col]
        return self.numeric_strategy if is_numeric else self.categorical_strategy

    @staticmethod
    def _compute_fill_value(series, strategy):
        """Compute a single scalar fill value for a pandas Series given a strategy name."""
        if strategy == 'mean':
            return series.mean()
        elif strategy == 'median':
            return series.median()
        elif strategy == 'most_frequent':
            mode = series.mode(dropna=True)
            return mode.iloc[0] if not mode.empty else np.nan
        else:
            raise ValueError(
                f"Unknown strategy '{strategy}'. Expected one of "
                "'mean', 'median', 'most_frequent'."
            )

    def fit(self, X, y=None):
        """
        Learn and store the fill value for every column in X.

        Parameters
        ----------
        X : pandas.DataFrame
            Training data. Must be a DataFrame -- dtypes and column names
            are inspected to auto-detect numeric vs. categorical columns.
        y : ignored
            Present only for scikit-learn API / Pipeline compatibility.

        Returns
        -------
        self : CustomImputer
            The fitted imputer (so calls can be chained, e.g.
            `CustomImputer().fit(X_train).transform(X_train)`).
        """
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)

        self.fill_values_ = {}
        self.numeric_cols_ = []
        self.categorical_cols_ = []
        self.indicator_cols_ = []

        for col in X.columns:
            is_numeric = ptypes.is_numeric_dtype(X[col])
            (self.numeric_cols_ if is_numeric else self.categorical_cols_).append(col)

            strategy = self._strategy_for(col, is_numeric)
            self.fill_values_[col] = self._compute_fill_value(X[col], strategy)

            if X[col].isna().any():
                self.indicator_cols_.append(col)

        self.feature_names_in_ = list(X.columns)
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        """
        Fill missing values in X using the statistics learned during `fit`.

        `transform` never recomputes anything from X itself -- it only reads
        `self.fill_values_`, which was populated by `fit`. This is what
        makes it safe to call on the training split, the test split, or
        brand-new data without ever leaking information between them.

        Parameters
        ----------
        X : pandas.DataFrame
            Data to transform. Should have the same columns seen in `fit`
            (columns not seen during `fit` are left untouched -- see the
            reflection-question demo in Section 6 for what that means).

        Returns
        -------
        X_transformed : pandas.DataFrame
            A COPY of X (the original is never mutated) with missing values
            filled in, plus one '<column>_was_missing' indicator column per
            column that had missing values during `fit`, if
            `add_missing_indicator=True`.

        Raises
        ------
        sklearn.exceptions.NotFittedError
            If `transform` is called before `fit`.
        """
        check_is_fitted(self, attributes=['fill_values_'])

        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X)
        X = X.copy()

        # Add indicators BEFORE filling, and BEFORE X's own NaNs are touched,
        # so they reflect this split's actual missingness pattern.
        if self.add_missing_indicator:
            for col in self.indicator_cols_:
                if col in X.columns:
                    X[f'{col}_was_missing'] = X[col].isna().astype(int)

        for col, fill_value in self.fill_values_.items():
            if col in X.columns:
                X[col] = X[col].fillna(fill_value)

        return X

## 2. Load and Split the Dataset

`student_id` is a row identifier, not a predictive feature, so it's set as the index rather than treated
as a column to impute. `final_score` is the prediction target, so it's excluded from the feature set `X`
that gets fitted/transformed (best practice: don't run feature-engineering steps on the label). Neither
choice affects correctness here since both columns are fully populated anyway.

In [3]:
df = pd.read_csv('ASSIGNMENTS/data/student_performance_raw.csv')
df = df.set_index('student_id')

print(f"Shape: {df.shape}")
print()
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Shape: (600, 16)

Missing values per column:
weekly_study_hours    36
income_bracket        30
prev_exam_score       25
mock_test_3           21
feedback_text         68
dtype: int64


In [4]:
feature_cols = [c for c in df.columns if c != 'final_score']

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (480, 15)
X_test shape:  (120, 15)


## 3. Fit on Training Data Only

`fit` is called on `X_train` alone -- never on `df`, never on `X_test`, and never on the two concatenated.
See Reflection Question 1 for why.

In [5]:
imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    add_missing_indicator=True,
)
imputer.fit(X_train)

print("Auto-detected numeric columns:")
print(imputer.numeric_cols_)
print()
print("Auto-detected categorical/text columns:")
print(imputer.categorical_cols_)
print()
print("Columns with missing values in the training split (will get *_was_missing flags):")
print(imputer.indicator_cols_)
print()
print("Learned fill values:")
for col in imputer.indicator_cols_:
    print(f"  {col!r}: {imputer.fill_values_[col]!r}")

Auto-detected numeric columns:
['city_tier', 'age', 'attendance_pct', 'weekly_study_hours', 'prev_exam_score', 'mock_test_1', 'mock_test_2', 'mock_test_3', 'doubt_sessions_attended']

Auto-detected categorical/text columns:
['city', 'course', 'batch_type', 'enrollment_date', 'income_bracket', 'feedback_text']

Columns with missing values in the training split (will get *_was_missing flags):
['weekly_study_hours', 'income_bracket', 'prev_exam_score', 'mock_test_3', 'feedback_text']

Learned fill values:
  'weekly_study_hours': np.float64(5.0)
  'income_bracket': '5-10L'
  'prev_exam_score': np.float64(66.1)
  'mock_test_3': np.float64(71.3)
  'feedback_text': 'Need more practice sheets for weak topics'


## 4. Transform Both Splits

The *same* fitted imputer is used to transform both `X_train` and `X_test` -- the test split only ever
has values looked up from `imputer.fill_values_`, which were learned exclusively from `X_train`.

In [6]:
X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)

new_indicator_cols = [f'{c}_was_missing' for c in imputer.indicator_cols_]
print("New indicator columns added:", new_indicator_cols)
print()
X_train_imputed[['weekly_study_hours', 'weekly_study_hours_was_missing',
                  'prev_exam_score', 'prev_exam_score_was_missing']].head(8)

New indicator columns added: ['weekly_study_hours_was_missing', 'income_bracket_was_missing', 'prev_exam_score_was_missing', 'mock_test_3_was_missing', 'feedback_text_was_missing']



,weekly_study_hours,weekly_study_hours_was_missing,prev_exam_score,prev_exam_score_was_missing
student_id,,,,
1551,2.0,0,56.5,0
1166,5.0,1,57.8,0
1367,6.2,0,45.1,0
1562,1.1,0,54.3,0
1575,2.8,0,78.8,0
1352,12.0,0,61.0,0
1535,1.2,0,78.0,0
1481,11.3,0,66.1,0


## 5. Verify No Missing Values Remain

Checked on the columns the imputer actually touched (`imputer.indicator_cols_`), on both splits.

In [7]:
imputed_cols = imputer.indicator_cols_

train_missing_after = X_train_imputed[imputed_cols].isna().sum().sum()
test_missing_after = X_test_imputed[imputed_cols].isna().sum().sum()

assert train_missing_after == 0, "Missing values remain in the imputed training columns!"
assert test_missing_after == 0, "Missing values remain in the imputed test columns!"

# Also confirm no *_was_missing indicator column itself contains NaN.
assert X_train_imputed[new_indicator_cols].isna().sum().sum() == 0
assert X_test_imputed[new_indicator_cols].isna().sum().sum() == 0

print(f"Remaining missing values in imputed columns -- train: {train_missing_after}, test: {test_missing_after}")
print("All assertions passed: no missing values remain in the imputed columns on either split.")

Remaining missing values in imputed columns -- train: 0, test: 0
All assertions passed: no missing values remain in the imputed columns on either split.


## 6. Before/After Statistics (Training Split)

Mean and standard deviation of every numeric column, before imputation (raw `X_train`, which `transform`
never mutates in place) and after (`X_train_imputed`).

In [8]:
rows = []
for col in imputer.numeric_cols_:
    rows.append({
        'column': col,
        'n_missing': int(X_train[col].isna().sum()),
        'mean_before': X_train[col].mean(),
        'mean_after': X_train_imputed[col].mean(),
        'std_before': X_train[col].std(),
        'std_after': X_train_imputed[col].std(),
    })

stats_df = pd.DataFrame(rows).set_index('column').round(3)
stats_df

,n_missing,mean_before,mean_after,std_before,std_after
column,,,,,
city_tier,0,1.402,1.402,0.491,0.491
age,0,16.956,16.956,1.731,1.731
attendance_pct,0,78.546,78.546,13.460,13.460
weekly_study_hours,25,6.064,6.009,4.350,4.242
prev_exam_score,23,65.825,65.838,14.371,14.022
mock_test_1,0,65.945,65.945,16.566,16.566
mock_test_2,0,67.886,67.886,18.109,18.109
mock_test_3,15,70.702,70.720,19.077,18.777
doubt_sessions_attended,0,4.004,4.004,2.026,2.026


**What changed:** columns with zero missing values (`city_tier`, `age`, `attendance_pct`,
`mock_test_1`, `mock_test_2`, `doubt_sessions_attended`) are, correctly, completely unchanged -- the
imputer only ever touches cells that were actually `NaN`.

For the three columns that *did* have missing values (`weekly_study_hours`, `prev_exam_score`,
`mock_test_3`), the mean barely moves (median imputation inserts a single central value repeatedly, which
pulls the mean only slightly, and only when the median and the original mean already differed). The
standard deviation, however, drops for all three -- inserting the *same* constant value in place of
several different (unknown) real values necessarily removes variance from the column. This is the classic,
unavoidable side effect of simple mean/median imputation: it quietly makes the imputed column look less
spread out than the true underlying data, which is worth remembering before feeding it to any downstream
model that cares about variance (e.g. anything using standard errors or confidence intervals).

## 7. Sanity Check Against scikit-learn's `SimpleImputer`

If `CustomImputer`'s core logic is correct, its learned fill values should match `SimpleImputer`'s
`statistics_` exactly, for both a numeric strategy and the categorical `most_frequent` strategy.

In [9]:
numeric_missing_cols = [c for c in imputer.indicator_cols_ if c in imputer.numeric_cols_]

sk_numeric_imputer = SimpleImputer(strategy='median')
sk_numeric_imputer.fit(X_train[numeric_missing_cols])

print(f"{'column':22s}{'CustomImputer':>16s}{'SimpleImputer':>16s}{'match':>8s}")
for col, sk_value in zip(numeric_missing_cols, sk_numeric_imputer.statistics_):
    custom_value = imputer.fill_values_[col]
    matches = np.isclose(custom_value, sk_value)
    print(f"{col:22s}{custom_value:16.4f}{sk_value:16.4f}{str(matches):>8s}")
    assert matches, f"Mismatch for {col}!"

print()
print("All numeric fill values match SimpleImputer(strategy='median') exactly.")

column                   CustomImputer   SimpleImputer   match
weekly_study_hours              5.0000          5.0000    True
prev_exam_score                66.1000         66.1000    True
mock_test_3                    71.3000         71.3000    True

All numeric fill values match SimpleImputer(strategy='median') exactly.


In [10]:
categorical_missing_cols = [c for c in imputer.indicator_cols_ if c in imputer.categorical_cols_]

sk_categorical_imputer = SimpleImputer(strategy='most_frequent')
sk_categorical_imputer.fit(X_train[categorical_missing_cols])

for col, sk_value in zip(categorical_missing_cols, sk_categorical_imputer.statistics_):
    custom_value = imputer.fill_values_[col]
    matches = (custom_value == sk_value)
    print(f"column: {col}")
    print(f"  CustomImputer fill value : {custom_value!r}")
    print(f"  SimpleImputer fill value : {sk_value!r}")
    print(f"  match: {matches}")
    assert matches, f"Mismatch for {col}!"

print()
print("All categorical fill values match SimpleImputer(strategy='most_frequent') exactly.")

column: income_bracket
  CustomImputer fill value : '5-10L'
  SimpleImputer fill value : '5-10L'
  match: True
column: feedback_text
  CustomImputer fill value : 'Need more practice sheets for weak topics'
  SimpleImputer fill value : 'Need more practice sheets for weak topics'
  match: True

All categorical fill values match SimpleImputer(strategy='most_frequent') exactly.


# Result

After Going through the result , the `CustomImputer` and `SimpleImputer` a match in both `categorical` and `numerical` data

## 8. Guard Rail: `transform()` Before `fit()`

`check_is_fitted` should raise a clear `NotFittedError` if someone tries to `transform` before `fit`.

In [11]:
unfitted_imputer = CustomImputer()
try:
    unfitted_imputer.transform(X_test)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

NotFittedError: This CustomImputer instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.


## 9. Reflection Questions

### Q1. Why must `fit()` be called only on the training split, and never on the full dataset or the test split?

Fitting on anything beyond the training split lets information from the test set leak into the
preprocessing pipeline: the fill values (means, medians, modes) would then be partly derived from data the
model is supposed to have never seen. That makes any downstream evaluation metric overly optimistic,
because the "unseen" test data isn't really unseen -- it already influenced how missing values were
filled. Fitting only on `X_train` keeps the pipeline honest: the fill values reflect only what would
genuinely be knowable at training time, which is exactly what happens in production when real new data
arrives one row (or one batch) at a time, with no access to future data.

### Q2. `mock_test_3` is MNAR. Does mean/median imputation genuinely solve the problem? What does `add_missing_indicator` contribute that plain imputation does not?

No -- mean/median imputation does not solve an MNAR problem, it just papers over it. In `mock_test_3`,
values are missing *because of the value itself* (e.g. students who would have scored very low are the
ones whose score is missing), so filling those cells with the overall median assumes they'd look like a
typical, middling data point -- which is precisely what MNAR says is *not* true. There's no way to recover
the real missing values from the observed data alone, since nothing else in the dataset explains *why*
that specific score is missing. What `add_missing_indicator` contributes is not a better guess at the true
value, but a separate signal a downstream model can use directly: the binary `mock_test_3_was_missing`
flag lets a model learn "rows where this was missing tend to behave like X" independently of whatever
(likely wrong) number got imputed into `mock_test_3` itself -- which is the closest a purely tabular
approach can get to acknowledging that the missingness itself carries information.

### Q3. A brand-new column, entirely missing in the training data but present in the test data, is passed to your imputer. What does your current implementation do -- and what SHOULD a production-grade version do instead?

As demonstrated below: `fit()` only ever iterates over `X.columns` as seen during fitting, so a column
that wasn't part of the training DataFrame never gets an entry in `self.fill_values_`. In `transform()`,
the fill loop is keyed off `self.fill_values_.items()`, so a column with no stored fill value is silently
skipped -- it passes through completely untouched, `NaN`s and all. That's a dangerous silent failure: a
downstream model would either crash on the unexpected `NaN`s or, worse, silently mishandle them. A
production-grade version should instead treat an unseen column as a schema violation: at minimum, raise a
clear warning (or a hard error, depending on how strict the pipeline needs to be) naming the unexpected
column(s), and require an explicit, documented policy for handling them -- e.g. drop them, impute with a
sensible global fallback, or refuse to proceed -- rather than quietly leaving missing values unfilled.

In [12]:
demo_train = X_train.copy()
demo_test = X_test.copy()
demo_test['new_unseen_column'] = np.random.choice([1.0, 2.0, np.nan], size=len(demo_test))

demo_imputer = CustomImputer()
demo_imputer.fit(demo_train)
demo_result = demo_imputer.transform(demo_test)

print("Was 'new_unseen_column' seen during fit?", 'new_unseen_column' in demo_imputer.fill_values_)
print("Missing values in 'new_unseen_column' BEFORE transform:", demo_test['new_unseen_column'].isna().sum())
print("Missing values in 'new_unseen_column' AFTER transform: ", demo_result['new_unseen_column'].isna().sum())
print()
print("-> Confirmed: the current implementation leaves an unseen column's NaNs completely unfilled.")

Was 'new_unseen_column' seen during fit? False
Missing values in 'new_unseen_column' BEFORE transform: 41
Missing values in 'new_unseen_column' AFTER transform:  41

-> Confirmed: the current implementation leaves an unseen column's NaNs completely unfilled.


## 10. Bonus — Per-Column Strategy Override (+10%)

`column_overrides` lets individual columns use a different strategy than the dataset-wide default.
Demonstrated on two columns:

- `weekly_study_hours` (numeric): overridden to `'mean'`, while the dataset-wide default stays `'median'`
  — this should visibly change the fill value used.
- `income_bracket` (categorical): explicitly overridden to `'most_frequent'` — same as the categorical
  default, which confirms the override mechanism also works correctly for categorical columns, not just
  numeric ones.

In [13]:
bonus_imputer = CustomImputer(
    numeric_strategy='median',
    categorical_strategy='most_frequent',
    add_missing_indicator=True,
    column_overrides={
        'weekly_study_hours': 'mean',
        'income_bracket': 'most_frequent',
    },
)
bonus_imputer.fit(X_train)
bonus_test_imputed = bonus_imputer.transform(X_test)

print("weekly_study_hours -- overridden strategy: 'mean' (dataset-wide default is 'median')")
print(f"  dataset-wide default (median) would give : {X_train['weekly_study_hours'].median():.4f}")
print(f"  actual column mean                        : {X_train['weekly_study_hours'].mean():.4f}")
print(f"  fill value bonus_imputer actually used     : {bonus_imputer.fill_values_['weekly_study_hours']:.4f}")
assert np.isclose(bonus_imputer.fill_values_['weekly_study_hours'], X_train['weekly_study_hours'].mean())
assert not np.isclose(bonus_imputer.fill_values_['weekly_study_hours'], X_train['weekly_study_hours'].median())
print("  -> matches the column mean, NOT the dataset-wide median default. Override works.")
print()

print("income_bracket -- explicitly overridden strategy: 'most_frequent' (same as categorical default)")
print(f"  fill value used: {bonus_imputer.fill_values_['income_bracket']!r}")
assert bonus_imputer.fill_values_['income_bracket'] == X_train['income_bracket'].mode().iloc[0]
print("  -> matches the column mode. Override mechanism works for categorical columns too.")
print()

assert bonus_test_imputed['weekly_study_hours'].isna().sum() == 0
assert bonus_test_imputed['income_bracket'].isna().sum() == 0
print("Verified: both overridden columns are fully imputed on the test split, with no missing values left.")

weekly_study_hours -- overridden strategy: 'mean' (dataset-wide default is 'median')
  dataset-wide default (median) would give : 5.0000
  actual column mean                        : 6.0642
  fill value bonus_imputer actually used     : 6.0642
  -> matches the column mean, NOT the dataset-wide median default. Override works.

income_bracket -- explicitly overridden strategy: 'most_frequent' (same as categorical default)
  fill value used: '5-10L'
  -> matches the column mode. Override mechanism works for categorical columns too.

Verified: both overridden columns are fully imputed on the test split, with no missing values left.


## Summary

- `CustomImputer` follows the scikit-learn `BaseEstimator`/`TransformerMixin` API (`fit` returns `self`,
  `transform` returns a copy), auto-detects numeric vs. categorical columns, and raises a clear error if
  `transform` is called before `fit`.
- It was fit strictly on the training split and used to transform both splits, with zero missing values
  left afterward in either one.
- Its learned fill values match `SimpleImputer` exactly, for both the numeric and categorical strategies.
- The missing-value indicator columns give a downstream model a way to use the *fact* of missingness as a
  signal in its own right -- most relevant for the MNAR column, `mock_test_3`, where plain imputation alone
  cannot recover what the true values would have been.
- The bonus `column_overrides` parameter lets individual columns depart from the dataset-wide default
  strategy, demonstrated on one numeric and one categorical column.